# Skin Lesion Classifier — ANN Baseline on HAM10000

**Goal**: Train an Artificial Neural Network (ANN) on HAM10000 images as a baseline model.
This will later be compared against a CNN to evaluate whether convolutional layers improve
skin lesion classification performance.

**Dataset**: [HAM10000 (Skin Cancer MNIST)](https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000)
— 10,015 dermatoscopic images across 7 diagnostic categories.

**Model**: Fully-connected ANN with 2 hidden layers, taking flattened 64×64 RGB images as input.

---
## 0. Setup

In [ ]:
!pip install kagglehub -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import zipfile
from pathlib import Path

import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, recall_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)

print(f"TensorFlow version: {tf.__version__}")
print(f"Kagglehub version: {kagglehub.__version__}")

---
## 1. Download HAM10000 via KaggleHub

> **Note**: You need a Kaggle API token to download. On Colab, add it as a secret named `KAGGLE_API_TOKEN`.
> Alternatively, run `kagglehub.login()` and paste your token.

In [ ]:
print("Downloading HAM10000 from Kaggle...")
data_path = kagglehub.dataset_download('kmader/skin-cancer-mnist-ham10000')
print(f"Downloaded to: {data_path}")
print(f"Contents: {os.listdir(data_path)}")

---
## 2. Metadata & EDA

In [ ]:
metadata_path = os.path.join(data_path, 'HAM10000_metadata.csv')
df = pd.read_csv(metadata_path)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

In [ ]:
dx_labels = {
    'nv': 'Melanocytic nevi',
    'mel': 'Melanoma',
    'bkl': 'Benign keratosis',
    'bcc': 'Basal cell carcinoma',
    'akiec': 'Actinic keratosis',
    'vasc': 'Vascular lesions',
    'df': 'Dermatofibroma'
}

dx_counts = df['dx'].value_counts()
dx_pct = df['dx'].value_counts(normalize=True) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B', '#44BBA4', '#E94F37']
full_labels = [dx_labels[dx] for dx in dx_counts.index]

bars = ax1.barh(full_labels, dx_counts.values, color=colors)
ax1.set_title('Diagnosis Counts', fontsize=14, fontweight='bold')
ax1.set_xlabel('Number of Images')
for i, (bar, val) in enumerate(zip(bars, dx_counts.values)):
    pct = dx_pct.iloc[i]
    ax1.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
             f"{val} ({pct:.1f}%)", va='center', fontsize=9)

wedges, texts = ax2.pie(
    dx_counts.values,
    labels=[f"{l}\n({p:.1f}%)" for l, p in zip(full_labels, dx_pct.values)],
    colors=colors, startangle=90, textprops={'fontsize': 9}
)
ax2.set_title('Class Proportions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nClass distribution:\n{dx_counts.to_string()}")
print(f"\nMajority class (nv): {dx_counts['nv']/len(df)*100:.1f}% — severe imbalance")

In [ ]:
average_age = df.groupby('dx')['age'].mean().sort_values(ascending=False).reset_index()
average_age['diagnosis'] = average_age['dx'].map(dx_labels)

plt.figure(figsize=(12, 6))
plot = sns.barplot(x='diagnosis', y='age', data=average_age, palette='magma')
plt.title('Average Age by Diagnosis', fontsize=15, fontweight='bold', pad=20)
plt.xlabel('Skin Lesion Type', fontsize=12)
plt.ylabel('Average Age (Years)', fontsize=12)
plt.xticks(rotation=45, ha='right')
for container in plot.containers:
    plot.bar_label(container, fmt='%.2f', padding=2)
plt.tight_layout()
plt.show()

In [ ]:
df_clean = df.dropna(subset=['sex'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(ax=axes[0], x='sex', data=df_clean,
              palette=['skyblue', 'lightpink', '#A9A9A9'])
axes[0].set_title('Total Count by Gender', fontsize=14)
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Count')

sex_dx_prop = pd.crosstab(df['dx'], df['sex'], normalize='index') * 100
sex_dx_prop.index = [dx_labels[d] for d in sex_dx_prop.index]
sex_dx_prop.plot(kind='bar', stacked=True, ax=axes[1],
                 color=['#4C72B0', '#DD8452', '#A9A9A9'])
axes[1].set_title('Sex Proportion for Each Skin Lesion', fontsize=14)
axes[1].set_xlabel('Skin Lesion')
axes[1].set_ylabel('Percentage')
axes[1].tick_params(axis='x', rotation=45)
plt.setp(axes[1].get_xticklabels(), ha='right')
axes[1].legend(title='Sex')
plt.tight_layout()
plt.show()

In [ ]:
localization_dx_prop = pd.crosstab(df['localization'], df['dx'], normalize='index')

fig, ax = plt.subplots(figsize=(14, 7))
localization_dx_prop.plot(kind='bar', stacked=True, ax=ax, cmap='tab20', width=0.8)
ax.set_xlabel('Localization', fontsize=12)
ax.set_ylabel('Proportion of Lesion Types', fontsize=12)
ax.set_title('Distribution of Lesion Types by Body Location', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=45)
plt.setp(ax.get_xticklabels(), ha='right')
ax.legend(title='Lesion Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

---
## 3. Load & Preprocess Images

- Resize to 64×64 RGB (balance between detail and memory)
- Normalize pixel values to [0, 1]
- Map `dx` labels to integer class indices

In [ ]:
IMG_SIZE = 64

part1_dir = os.path.join(data_path, 'HAM10000_images_part_1')
part2_dir = os.path.join(data_path, 'HAM10000_images_part_2')

# If zipped, extract
for part_dir, zip_name in [(part1_dir, 'HAM10000_images_part_1.zip'),
                           (part2_dir, 'HAM10000_images_part_2.zip')]:
    if not os.path.isdir(part_dir):
        zip_path = os.path.join(data_path, zip_name)
        if os.path.exists(zip_path):
            print(f"Extracting {zip_name}...")
            with zipfile.ZipFile(zip_path, 'r') as zf:
                zf.extractall(data_path)

print(f"Part 1 contains: {len(os.listdir(part1_dir))} files")
print(f"Part 2 contains: {len(os.listdir(part2_dir))} files")

In [ ]:
def load_images(df, part1_dir, part2_dir, img_size=IMG_SIZE):
    """Load HAM10000 images, resized and normalized."""
    images = []
    labels = []
    missing = 0

    for _, row in df.iterrows():
        img_name = f"{row['image_id']}.jpg"
        img_path = os.path.join(part1_dir, img_name)
        if not os.path.exists(img_path):
            img_path = os.path.join(part2_dir, img_name)

        if not os.path.exists(img_path):
            missing += 1
            continue

        img = keras.preprocessing.image.load_img(
            img_path, target_size=(img_size, img_size)
        )
        img = keras.preprocessing.image.img_to_array(img) / 255.0
        images.append(img)
        labels.append(row['dx'])

    if missing:
        print(f"Warning: {missing} images not found")

    return np.array(images, dtype=np.float32), np.array(labels)

In [ ]:
print("Loading HAM10000 images (64x64 RGB)...")
X, y_raw = load_images(df, part1_dir, part2_dir)
print(f"Loaded {X.shape[0]} images, shape: {X.shape}")
print(f"Pixel range: [{X.min():.3f}, {X.max():.3f}]")

In [ ]:
le = LabelEncoder()
y = le.fit_transform(y_raw)
class_names = le.classes_
num_classes = len(class_names)

print(f"Classes ({num_classes}): {list(class_names)}")
for i, cls in enumerate(class_names):
    count = (y == i).sum()
    print(f"  {i}: {cls} — {count} samples ({count/len(y)*100:.1f}%)")

In [ ]:
# Sample of loaded images
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for i, ax in enumerate(axes.flat):
    idx = np.random.randint(len(X))
    ax.imshow(X[idx])
    ax.set_title(class_names[y[idx]], fontsize=10)
    ax.axis('off')
plt.suptitle('Sample HAM10000 Images (64x64)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Train/Validation/Test Split

Stratified 72/8/20 split to preserve class distribution across all splits.

In [ ]:
# First split: separate test set (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Second split: train (90% of 80% = 72%) and validation (10% of 80% = 8%)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1, random_state=42, stratify=y_temp
)

print(f"Train:       {X_train.shape[0]:,} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation:  {X_val.shape[0]:,} ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test:        {X_test.shape[0]:,} ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"Total:       {len(X):,}")

In [ ]:
# Verify stratification
def dist(y_data, name):
    counts = pd.Series(y_data).value_counts().sort_index()
    pcts = counts / counts.sum() * 100
    return pd.DataFrame({f'{name}_count': counts, f'{name}_pct': pcts})

pd.concat([
    dist(y_train, 'train'),
    dist(y_val, 'val'),
    dist(y_test, 'test')
], axis=1).fillna(0)

---
## 5. Build ANN

Fully-connected network:
- Input: 64×64×3 = 12,288 flattened pixels
- Hidden 1: 128 neurons, ReLU, Dropout 30%
- Hidden 2: 64 neurons, ReLU, Dropout 30%
- Output: 7 neurons, Softmax

In [ ]:
model = keras.Sequential([
    layers.Flatten(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                   name='flatten'),
    layers.Dense(128, activation='relu', name='dense_128'),
    layers.Dropout(0.3, name='dropout_1'),
    layers.Dense(64, activation='relu', name='dense_64'),
    layers.Dropout(0.3, name='dropout_2'),
    layers.Dense(num_classes, activation='softmax', name='output')
], name='ANN_Baseline')

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Compute class weights to handle severe imbalance
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

print("Class weights:")
for i, (cls, w) in enumerate(zip(class_names, class_weights)):
    count = (y_train == i).sum()
    print(f"  {cls:8s}: {count:4d} samples → weight {w:.2f}")

---
## 6. Train ANN

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-6
        )
    ],
    verbose=1
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['loss'], label='Train', linewidth=2)
ax1.plot(history.history['val_loss'], label='Validation', linewidth=2)
ax1.set_title('Model Loss', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['accuracy'], label='Train', linewidth=2)
ax2.plot(history.history['val_accuracy'], label='Validation', linewidth=2)
ax2.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_epoch = np.argmin(history.history['val_loss'])
print(f"Best epoch: {best_epoch + 1}")
print(f"Best val_loss: {history.history['val_loss'][best_epoch]:.4f}")
print(f"Best val_accuracy: {history.history['val_accuracy'][best_epoch]:.4f}")

---
## 7. Evaluation on Test Set

In [ ]:
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

print("Classification Report:")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=class_names,
                            digits=3))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix — Test Set', fontsize=14, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

In [ ]:
# Normalized confusion matrix
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(10, 8))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Normalized Confusion Matrix — Test Set', fontsize=14, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
class_recall = recall_score(y_test, y_pred, average=None)

print("=" * 55)
print("ANN Baseline — Final Results")
print("=" * 55)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Mean Recall:   {class_recall.mean():.4f}")
print()
print("Per-class Recall:")
print("-" * 30)
for i, cls in enumerate(class_names):
    print(f"  {cls:8s}: {class_recall[i]:.3f}")
print()
print("Note: Recall is our primary metric (per proposal).")
print(f"Majority class (nv) dominates with {dx_counts['nv']/len(df)*100:.1f}% of data.")
print("Low recall on rare classes is expected for a simple ANN baseline.")